In [ ]:
!pip install pandas numpy matplotlib seaborn plotly

In [ ]:
import pandas as pd

url = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"

# Load only first 200,000 rows (safe size)
df = pd.read_csv(url, sep='\t', nrows=200000, low_memory=False)

df.head()

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,54,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1582569031,2020-02-24T18:30:31Z,1733085204,2024-12-01T20:33:24Z,NaN,1.740205e+09,2025-02-22T06:23:42Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1673620307,2023-01-13T14:31:47Z,1750061386,2025-06-16T08:09:46Z,bodysupport,1.750061e+09,2025-06-16T08:09:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,114,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1580066482,2020-01-26T19:21:22Z,1751035658,2025-06-27T14:47:38Z,teolemon,1.751036e+09,2025-06-27T14:47:38Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,431,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1714301712,2024-04-28T10:55:12Z,1714301721,2024-04-28T10:55:21Z,kiliweb,1.714302e+09,2024-04-28T10:55:21Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,105,http://world-en.openfoodfacts.org/product/0000...,kiliweb,1572117743,2019-10-26T19:22:23Z,1738073570,2025-01-28T14:12:50Z,NaN,1.743653e+09,2025-04-03T04:11:36Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.shape

(200000, 210)

In [ ]:
df.columns

Index(['code', 'url', 'creator', 'created_t', 'created_datetime',
       'last_modified_t', 'last_modified_datetime', 'last_modified_by',
       'last_updated_t', 'last_updated_datetime',
       ...
       'choline_100g', 'phylloquinone_100g', 'beta-glucan_100g',
       'inositol_100g', 'carnitine_100g', 'sulphate_100g', 'nitrate_100g',
       'acidity_100g', 'carbohydrates-total_100g', 'water_100g'],
      dtype='object', length=210)

In [ ]:
df[['product_name', 'sugars_100g', 'proteins_100g', 'categories_tags']].head(10)

,product_name,sugars_100g,proteins_100g,categories_tags
0,Limonade artisanale a la rose,NaN,NaN,NaN
1,M&amp;M white,NaN,NaN,NaN
2,Chocolate n3,NaN,NaN,NaN
3,Pâte de fruits,NaN,NaN,NaN
4,Paleta gran reserva - Sierra nevada-,NaN,NaN,"en:beverages-and-beverages-preparations,en:bev..."
5,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN
8,Confiture extra citron de Menton,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN


In [ ]:
df = df[['product_name', 'sugars_100g', 'proteins_100g', 'categories_tags', 'ingredients_text']]

In [ ]:
df = df.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])

In [ ]:
df = df[
    (df['sugars_100g'] >= 0) & (df['sugars_100g'] <= 100) &
    (df['proteins_100g'] >= 0) & (df['proteins_100g'] <= 100)
]

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
def assign_category(tags):
    if pd.isna(tags):
        return "Other"

    tags = tags.lower()

    if "chocolate" in tags or "sweets" in tags:
        return "Sweets"
    elif "biscuits" in tags or "cookies" in tags:
        return "Biscuits"
    elif "beverages" in tags or "drinks" in tags:
        return "Beverages"
    elif "snacks" in tags:
        return "Snacks"
    elif "cereals" in tags or "breakfast" in tags:
        return "Cereals"
    else:
        return "Other"

In [ ]:
df['primary_category'] = df['categories_tags'].apply(assign_category)

In [ ]:
df['primary_category'].value_counts()

,count
primary_category,
Other,22478
Beverages,7645
Snacks,1311
Biscuits,930
Sweets,804
Cereals,79


In [ ]:
df[['sugars_100g', 'proteins_100g']].describe()

,sugars_100g,proteins_100g
count,33247.000000,33247.000000
mean,12.961551,9.721669
std,17.740572,12.737228
min,0.000000,0.000000
25%,1.333333,2.500000
50%,4.705882,6.670000
75%,17.699108,11.111111
max,100.000000,100.000000


In [ ]:
import plotly.express as px

fig = px.scatter(
    df.sample(5000),  # sample for performance
    x='sugars_100g',
    y='proteins_100g',
    color='primary_category',
    title='Sugar vs Protein Distribution'
)

fig.show()

In [ ]:
gap_df = df[
    (df['proteins_100g'] > 10) &
    (df['sugars_100g'] < 5)
]

In [ ]:
gap_df.shape

(6604, 6)

In [ ]:
from collections import Counter

ingredients = gap_df['ingredients_text'].dropna().str.lower()

words = []
for item in ingredients:
    words.extend(item.split(','))

common = Counter(words).most_common(10)

common

[(' salt', 1705),
 (' water', 793),
 (' sugar', 518),
 (' niacin', 438),
 (' yeast', 355),
 (' dextrose', 284),
 (' riboflavin', 267),
 (' enzymes', 266),
 (' spices', 259),
 (' sea salt', 218)]

In [ ]:
df.to_csv("cleaned_data.csv", index=False)